# Custom Level 1 Post-Analysis

Copy of `05_compass_level_1_post_analysis`, extended with the same custom `ZONES`/`ZONE_SETS`/`GENOTYPE_SETS` system built in `02b_custom_task_performance_analysis` -- no need for a separate "04b": the HMM fitting in `04_compass_level_1` doesn't depend on maze zones or genotype groupings at all, so its `data_with_states.csv` output is reused as-is.

In [ ]:
# autoreload
%load_ext autoreload
%autoreload 2

# Load Project

In [ ]:
from pathlib import Path
import pandas as pd
from compass_labyrinth import load_project


project_path = Path(r'D:\Gladstone Dropbox\Palop Lab\Nina\Experiments\For Level 1 CoMPASS States\New_Reward_E3E4NLF\compass_analysis')

# Import config and metadata
config, cohort_metadata = load_project(project_path=project_path)

### Load the HMM results

In [ ]:
df_hmm = pd.read_csv(project_path / "results" / "compass_level_1" / "data_with_states.csv")
df_hmm

### State Representation: Characteristics

- State 0 --> Stationary (excluded from HMM fitting -- step too small to reliably estimate turn angle, see notebook 04)
- State 1 --> Low Step Length, High Turn Angle --> "Surveillance" state (Red)
- State 2 --> High Step Length, Low Turn Angle --> "Ambulatory" state (Blue)

`MERGE_STATIONARY_INTO_SURVEILLANCE` (set in the "Define Custom Zones and Genotypes" cell below) controls whether state 0 gets folded into state 1 before any plot below runs -- set it to `False` to keep all three states separate.

## Define Custom Zones and Genotypes

Same `ZONES`/`ZONE_SETS`/`GENOTYPE_SETS`/`build_genotype_groups()` system as `02b` -- it's the same 10x10 maze, so the zone cell-number definitions are identical. `build_zone_column()` is new here: it adds a `custom_zone` column to a copy of `df_hmm` by `grid_number` membership (same overlap-safe, order-preserving approach as `02b`'s `build_zone_pivot()`), for use with `compute_state_probability()`'s `column_of_interest` (Plot 2).

Plot 3 (`compute_node_state_medians_over_time`) and Plot 4 (`compute_surveillance_probabilities`) don't need that column -- they already take a `node_type_mapping` dict directly (added alongside this notebook), so pass `node_type_mapping=ZONES` and any two `ZONES` keys as `decision_nodes`/`nondecision_nodes` to compare any custom zone pair, instead of only the fixed `constants.NODE_TYPE_MAPPING` keys.

`MERGE_STATIONARY_INTO_SURVEILLANCE` is a similar on/off toggle for HMM states: `True` (default) merges state 0 (Stationary) into state 1 (Surveillance) via `merge_hmm_states()`, applied to `df_hmm` once at the end of this cell -- every Plot 1-4 cell below reads the already-merged `df_hmm`, so no other cell needs to change when you flip it. Set it to `False` to keep the original 3-state system.

In [ ]:
# zone_name -> list of grid cell numbers (media/new_maze_layout_v2.svg). This
# is a mutually exclusive partition of all 100 cells. Organized in three
# groups: sequential maze segments (entry to exit), the 9 individual hairpin
# turns, then pooled/composite zones built from those. Identical to 02b's
# ZONES -- same maze, same cell numbering.
ZONES = {
    # --- Sequential maze segments, entry to exit ---
    "entry_zone": [0, 1],
    "entry_corridor": [2, 3, 4, 5, 6, 7],
    "column_zone": [
        12, 13, 14, 15, 16, 17,
        22, 23, 24, 25, 26, 27,
        32, 33, 34, 35, 36, 37,
        42, 43, 44, 45, 46, 47,
        52, 53, 54, 55, 56, 57,
        62, 63, 64, 65, 66, 67,
    ],
    "column_before_reward": [72, 73, 74, 75, 76, 77],
    "reward_zone": [70, 71, 80, 81],
    "column_after_reward": [82, 83, 84, 85, 86, 87],
    "overshoot_column": [92, 93, 94, 95, 96, 97],
    "terminal": [90, 91],

    # --- The 9 individual turns of the hairpin (T1-T9, alternating bottom/top):
    # T1 (bottom, col0-1), T2 (top, col1-2), T3 (bottom, col2-3), T4 (top,
    # col3-4), T5 (bottom, col4-5), T6 (top, col5-6), T7 (bottom, col6-7),
    # T8 (top, col7-8 -- this is exactly reward_zone above, no separate
    # alias needed), T9 (bottom, col8-9). Used by the "9_corners" ZONE_SET
    # below to give each turn its own line/row instead of the pooled
    # top_corners/bottom_corners groupings.
    "t1_corner": [8, 9, 18, 19],
    "t2_corner": [10, 11, 20, 21],
    "t3_corner": [28, 29, 38, 39],
    "t4_neutral": [30, 31, 40, 41],  # T4
    "t5_corner": [48, 49, 58, 59],
    "t6_corner": [50, 51, 60, 61],
    "t7_corner": [68, 69, 78, 79],
    "t9_corner": [88, 89, 98, 99],

    # --- Pooled/composite zones (groups of the segments/corners above) ---
    "top_corners": [10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61],  # T2, T4, T6
    "bottom_corners": [28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79],  # T3, T5, T7
    "all_corners": [
        10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61,
        28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79
    ],
    "aft_overshoot_column": [88, 89, 90, 91, 98, 99, 92, 93, 94, 95, 96, 97],

    # --- Row bands across columns 1-6 (05b-only zone set, not in 02b) ---
    # Individual rows across cols1-6, all 10 rows (row0 = top .. row9 =
    # bottom). Used by the "6_rows" (rows 0-2/7-9 only) and "10_rows" (all
    # 10) ZONE_SETs below.
    "row0_columns": [10, 20, 30, 40, 50, 60],
    "row1_columns": [11, 21, 31, 41, 51, 61],
    "row2_columns": [12, 22, 32, 42, 52, 62],
    "row3_columns": [13, 23, 33, 43, 53, 63],
    "row4_columns": [14, 24, 34, 44, 54, 64],
    "row5_columns": [15, 25, 35, 45, 55, 65],
    "row6_columns": [16, 26, 36, 46, 56, 66],
    "row7_columns": [17, 27, 37, 47, 57, 67],
    "row8_columns": [18, 28, 38, 48, 58, 68],
    "row9_columns": [19, 29, 39, 49, 59, 69],

    # Same cols1-6 row bands, paired up symmetrically (row0+row9, row1+row8,
    # row2+row7, row3+row6, row4+row5) -- used by the "3_rows" (first three
    # pairs only) and "5_rows" (all 5 pairs) ZONE_SETs below.
    "row0_9_columns": [10, 20, 30, 40, 50, 60, 19, 29, 39, 49, 59, 69],
    "row1_8_columns": [11, 21, 31, 41, 51, 61, 18, 28, 38, 48, 58, 68],
    "row2_7_columns": [12, 22, 32, 42, 52, 62, 17, 27, 37, 47, 57, 67],
    "row3_6_columns": [13, 23, 33, 43, 53, 63, 16, 26, 36, 46, 56, 66],
    "row4_5_columns": [14, 24, 34, 44, 54, 64, 15, 25, 35, 45, 55, 65],

    # Original 4 zones from 02_task_performance_analysis.ipynb (matches
    # constants.py's REGION_MAPPING exactly -- entry_zone(2) + reward_path(76)
    # + target_zone(4) + overshoot(18) = 100 cells). "target_zone" is the same
    # 4 cells as "reward_zone" above, just named to match notebook 02.
    "target_zone": [70, 71, 80, 81],
    "reward_path": (
        [2, 3, 4, 5, 6, 7]  # entry_corridor
        + [12, 13, 14, 15, 16, 17, 22, 23, 24, 25, 26, 27, 32, 33, 34, 35, 36, 37,
           42, 43, 44, 45, 46, 47, 52, 53, 54, 55, 56, 57, 62, 63, 64, 65, 66, 67]  # column_zone
        + [72, 73, 74, 75, 76, 77]  # column_before_reward
        + [8, 9, 18, 19]  # t1_corner
        + [10, 11, 20, 21, 30, 31, 40, 41, 50, 51, 60, 61]  # top_corners
        + [28, 29, 38, 39, 48, 49, 58, 59, 68, 69, 78, 79]  # bottom_corners
    ),
    "overshoot": (
        [82, 83, 84, 85, 86, 87]  # column_after_reward
        + [92, 93, 94, 95, 96, 97]  # overshoot_column
        + [88, 89, 90, 91, 98, 99]  # t9_corner + terminal
    ),
}

# Group zone names into the combinations used by the plots below (a zone can be in more than one set)
ZONE_SETS = {
    "5_columns": ["entry_corridor", "column_zone", "column_before_reward", "column_after_reward", "overshoot_column"],
    "4_corners": ["top_corners", "bottom_corners", "t4_neutral", "reward_zone"],
    "six_zones": ["all_corners", "column_zone", "column_before_reward", "reward_zone", "column_after_reward", "aft_overshoot_column"],
    "reward_zones": ["column_before_reward", "reward_zone", "column_after_reward"],
    "entry_reward": ["entry_zone", "entry_corridor", "top_corners", "bottom_corners", "column_zone", "reward_zone"],
    "original_4": ["entry_zone", "reward_path", "target_zone", "overshoot"],
    "except_entry": [
        "entry_corridor", "column_zone", "column_before_reward", "column_after_reward", "overshoot_column",
        "t1_corner", "top_corners", "bottom_corners", "reward_zone", "t9_corner", "terminal"
        ],
    "9_corners": [
        "t1_corner", "t2_corner", "t3_corner", "t4_neutral", "t5_corner",
        "t6_corner", "t7_corner", "reward_zone", "t9_corner",
    ],
    "6_rows": [
        "row0_columns", "row1_columns", "row2_columns",
        "row7_columns", "row8_columns", "row9_columns",
    ],
    "3_rows": ["row0_9_columns", "row1_8_columns", "row2_7_columns"],
    "10_rows": [
        "row0_columns", "row1_columns", "row2_columns", "row3_columns", "row4_columns",
        "row5_columns", "row6_columns", "row7_columns", "row8_columns", "row9_columns",
    ],
    "5_rows": ["row0_9_columns", "row1_8_columns", "row2_7_columns", "row3_6_columns", "row4_5_columns"],
}

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, except_entry, 9_corners, 6_rows, 3_rows, 10_rows, 5_rows, wt_nlf, apoe_pair

# set_name -> {label: [raw genotype strings that map to that label]}. A
# label mapping to a single raw genotype (e.g. "pair") keeps genotypes
# separate; a label mapping to several (e.g. "wt_nlf"'s "WT": ["E3", "E4"])
# pools them into one group. Both cases are used identically everywhere via
# build_genotype_groups() -- there's no separate "raw" vs "pooled" mechanism.
GENOTYPE_SETS = {
    "pair": {"E3": ["E3"], "E4": ["E4"]},
    "all": {"E3": ["E3"], "E4": ["E4"], "E3NLF": ["E3NLF"], "E4NLF": ["E4NLF"]},
    "apoe3_group": {"E3": ["E3"], "E3NLF": ["E3NLF"]},  # E3 vs E3NLF -- NLF's effect within the APOE3 background
    "apoe4_group": {"E4": ["E4"], "E4NLF": ["E4NLF"]},  # E4 vs E4NLF -- NLF's effect within the APOE4 background
    "apoe_pair": {"APOE3": ["E3", "E3NLF"], "APOE4": ["E4", "E4NLF"]},  # pooled by APOE allele
    "wt_nlf": {"WT": ["E3", "E4"], "NLF": ["E3NLF", "E4NLF"]},  # pooled by NLF modifier
}


def build_genotype_groups(df, set_name):
    """Relabel a dataframe's `genotype` column per GENOTYPE_SETS[set_name].

    Rows whose raw genotype isn't in any label's list are dropped; rows that
    are get `genotype` replaced by the label (e.g. E3 and E3NLF both become
    "APOE3" for set_name="apoe_pair"). Works the same whether the set keeps
    genotypes separate ("pair") or pools several into one label ("wt_nlf").

    set_name : key of GENOTYPE_SETS to use
    """
    mapping = GENOTYPE_SETS[set_name]
    genotype_lookup = {raw: label for label, raw_genotypes in mapping.items() for raw in raw_genotypes}
    df_grouped = df[df["genotype"].isin(genotype_lookup)].copy()
    df_grouped["genotype"] = df_grouped["genotype"].map(genotype_lookup)
    return df_grouped


def build_zone_column(df, zone_names, column_name="custom_zone"):
    """Add a `column_name` column labeling each row's zone by ZONES membership.

    zone_names : list of zone names (keys of ZONES) to include -- zones may
        overlap (share grid cells); a shared cell's rows are duplicated (once
        per zone it belongs to) rather than assigned to just one, same
        overlap-safe, order-preserving approach as 02b's build_zone_pivot().
        Rows whose grid_number isn't in any selected zone are dropped.
    column_name : name of the new column to add (default "custom_zone")
    """
    zones = {name: ZONES[name] for name in zone_names}
    labeled = []
    for zone, cells in zones.items():
        subset = df[df["grid_number"].isin(cells)].copy()
        subset[column_name] = zone
        labeled.append(subset)
    if not labeled:
        return df.iloc[0:0].assign(**{column_name: []})
    # Concat WITHOUT ignore_index, then sort_index to restore each session's
    # original chronological row order (see 02b's build_zone_pivot for why).
    return pd.concat(labeled).sort_index(kind="stable").reset_index(drop=True)


# Whether to merge HMM state 0 (Stationary) into state 1 (Surveillance)
# before any state-based analysis below -- set to False to keep the original
# 3-state system (0=Stationary, 1=Surveillance, 2=Ambulatory) instead.
MERGE_STATIONARY_INTO_SURVEILLANCE = True


def merge_hmm_states(df, merge=MERGE_STATIONARY_INTO_SURVEILLANCE):
    """Return a copy of df with hmm_state relabeled so 0 (Stationary) counts as 1 (Surveillance).

    No-op (besides the copy) if merge=False. Every Plot 1-4 function below
    keys off the literal 'hmm_state' column and a single int/exact-match
    filter, so applying this once (right here, before any plot cell runs)
    makes the merge apply everywhere automatically -- no need to touch each
    plot cell individually.
    """
    df = df.copy()
    if merge:
        df["hmm_state"] = df["hmm_state"].replace(0, 1)
    return df


df_hmm = merge_hmm_states(df_hmm)

# Plot 1: Heatmap Representations of HMM States

This workflow visualizes the spatial distribution of HMM state proportions over a grid-mapped maze using heatmaps. It overlays key regions such as decision and target nodes for the specified genotype.

### Recommended Use:
1. Ensure `df_hmm` contains columns: 'Genotype', 'Grid.Number', 'HMM_State', 'x', and 'y'.
2. Pick a `GENOTYPE_SET` from `GENOTYPE_SETS` (raw or pooled).
3. Set `grid_filename` to the corresponding shapefile for the session.
4. Use `compute_state_proportion()` to calculate per-grid HMM state proportions.
5. Load the grid geometry using `create_grid_geodata()`.
6. Use `map_points_to_grid()` and `sjoin()` to align state estimates with grid polygons.
7. Merge state proportions back into the grid using `merge_state_proportions_to_grid()`.
8. Plot the heatmap using `plot_grid_heatmap()` and highlight decision/target regions.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_proportion,
    create_grid_geodata,
    map_points_to_grid,
    merge_state_proportions_to_grid,
    plot_grid_heatmap,
)


# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, wt_nlf, apoe_pair
GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
grid_filename = "Session0001_grid.shp"
HMM_STATE = 1  # Fast / Ambulatory
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"

# Color scale bounds for every Plot 1 heatmap below (this cell + the 3 that
# reuse it). Narrow HEATMAP_VMIN up (e.g. 0.7) to spread contrast among
# values clustered near 1.
HEATMAP_VMIN = 0.3
HEATMAP_VMAX = 1.0

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)

# Step 1: Load session-specific grid geometry (shared across genotypes)
grid = create_grid_geodata(config=config, grid_filename=grid_filename)

for genotype_name in GENOTYPES:
    # Step 2: Compute HMM state proportions by grid
    state_df = compute_state_proportion(df=df_geno, genotype_name=genotype_name, hmm_state=HMM_STATE)
    if state_df.empty:
        print(f"No rows for genotype '{genotype_name}', skipping.")
        continue

    # Step 3: Map mean (x, y) points to grid polygons
    pointInPolys = map_points_to_grid(state_df, grid)

    # Step 4: Merge proportions with grid polygons
    grid_mapped = merge_state_proportions_to_grid(grid, state_df)

    # Step 5: Plot heatmap
    # highlight_grids dropped -- this maze has no branching, so there is no
    # "decision" node type to highlight (NODE_TYPE_MAPPING has no such key)
    plot_grid_heatmap(
        config=config,
        grid=grid_mapped,
        genotype_name=genotype_name,
        hmm_state=HMM_STATE,
        highlight_grids=None,
        target_grids="target_zone",
        vmin=HEATMAP_VMIN,
        vmax=HEATMAP_VMAX,
        filename_suffix=FILENAME_SUFFIX,
        figures_dirname="figures_lvl1_b/plot1_heatmap",
    )

print('Target Nodes highlighted in YELLOW')

### Stationary state heatmap (hmm_state = 0)

Unlike the original two-state (Surveillance / Ambulatory) design, this dataset has a third category -- frames excluded from HMM fitting because their step length was too small to reliably estimate a turning angle (see notebook 04). Worth its own spatial heatmap rather than leaving it out, since `compute_state_proportion` etc. are already generic to `hmm_state`.

In [ ]:
# Reuses df_geno/grid/GENOTYPES/FILENAME_SUFFIX/HEATMAP_VMIN/HEATMAP_VMAX from the cell above
for genotype_name in GENOTYPES:
    state_df_stationary = compute_state_proportion(df=df_geno, genotype_name=genotype_name, hmm_state=0)
    if state_df_stationary.empty:
        print(f"No rows for genotype '{genotype_name}', skipping.")
        continue

    grid_mapped_stationary = merge_state_proportions_to_grid(grid, state_df_stationary)

    plot_grid_heatmap(
        config=config,
        grid=grid_mapped_stationary,
        genotype_name=genotype_name,
        hmm_state=0,
        highlight_grids=None,
        target_grids="target_zone",
        vmin=HEATMAP_VMIN,
        vmax=HEATMAP_VMAX,
        filename_suffix=FILENAME_SUFFIX,
        figures_dirname="figures_lvl1_b/plot1_heatmap",
    )

print('Target Nodes highlighted in YELLOW')

## Heatmap representations of all genotypes

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_all_genotype_heatmaps


GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"
df_geno_all = build_genotype_groups(df_hmm, GENOTYPE_SET)

# Reuses HEATMAP_VMIN/HEATMAP_VMAX from the Plot 1 cell above
plot_all_genotype_heatmaps(
    config=config,
    df_hmm=df_geno_all,
    grid_filename=grid_filename,
    highlight_grids=None,
    target_grids="target_zone",
    hmm_state=1,
    cmap='YlOrRd',
    vmin=HEATMAP_VMIN,
    vmax=HEATMAP_VMAX,
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot1_heatmap_all_genotypes",
)

# Plot 1b: Heatmap Representations of Velocity

Same spatial heatmap mechanism as Plot 1, but colors each grid cell by mean velocity instead of HMM state proportion -- `compute_mean_velocity()` averages the `velocity` column (already present in `df_hmm`, no need for `ensure_velocity_column()`) per grid cell, and `plot_grid_heatmap()`/`plot_all_genotype_heatmaps()`'s `value_column`/`metric_label` parameters let them plot that instead of a state proportion.

### Recommended Use:
1. Pick a `GENOTYPE_SET` from `GENOTYPE_SETS`.
2. Use `compute_mean_velocity()` to calculate per-grid mean velocity.
3. Merge onto the grid with `merge_state_proportions_to_grid(..., value_col="velocity", target_col="mean_velocity")`.
4. Plot with `plot_grid_heatmap(..., value_column="mean_velocity", metric_label="Mean Velocity")`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_mean_velocity,
    create_grid_geodata,
    map_points_to_grid,
    merge_state_proportions_to_grid,
    plot_grid_heatmap,
)


GENOTYPE_SET = "wt_nlf"
GENOTYPES = list(GENOTYPE_SETS[GENOTYPE_SET].keys())
grid_filename = "Session0001_grid.shp"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"

# Color scale bounds for every Plot 1b heatmap below (this cell + the cell
# that reuses it) -- real per-cell mean velocity in this dataset ranges
# ~22-320, so these are round bounds around that, not [0, 1] like Plot 1.
VELOCITY_VMIN = 0
VELOCITY_VMAX = 300

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)

# Step 1: Load session-specific grid geometry (shared across genotypes)
grid = create_grid_geodata(config=config, grid_filename=grid_filename)

for genotype_name in GENOTYPES:
    # Step 2: Compute mean velocity by grid
    vel_df = compute_mean_velocity(df=df_geno, genotype_name=genotype_name)
    if vel_df.empty:
        print(f"No rows for genotype '{genotype_name}', skipping.")
        continue

    # Step 3: Map mean (x, y) points to grid polygons
    pointInPolys = map_points_to_grid(vel_df, grid)

    # Step 4: Merge velocity with grid polygons
    grid_mapped = merge_state_proportions_to_grid(grid, vel_df, value_col="velocity", target_col="mean_velocity")

    # Step 5: Plot heatmap
    plot_grid_heatmap(
        config=config,
        grid=grid_mapped,
        genotype_name=genotype_name,
        highlight_grids=None,
        target_grids="target_zone",
        cmap="YlGnBu",
        vmin=VELOCITY_VMIN,
        vmax=VELOCITY_VMAX,
        value_column="mean_velocity",
        metric_label="Mean Velocity",
        filename_stem="velocity_heatmap",
        filename_suffix=FILENAME_SUFFIX,
        figures_dirname="figures_lvl1_b/plot1b_velocity_heatmap",
    )

print('Target Nodes highlighted in YELLOW')

## Heatmap representations of velocity across all genotypes

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import compute_mean_velocity, plot_all_genotype_heatmaps


GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}"
df_geno_all = build_genotype_groups(df_hmm, GENOTYPE_SET)

# Reuses VELOCITY_VMIN/VELOCITY_VMAX from the Plot 1b cell above
plot_all_genotype_heatmaps(
    config=config,
    df_hmm=df_geno_all,
    grid_filename=grid_filename,
    highlight_grids=None,
    target_grids="target_zone",
    cmap="YlGnBu",
    vmin=VELOCITY_VMIN,
    vmax=VELOCITY_VMAX,
    compute_fn=lambda df, g: compute_mean_velocity(df, g),
    value_col="velocity",
    value_column="mean_velocity",
    metric_label="Mean Velocity",
    filename_stem="velocity_heatmap",
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot1b_velocity_heatmap_all_genotypes",
)

# Plot 2: Probability of Surveillance across Custom Zones

This workflow computes and visualizes the probability of HMM state occupancy across custom zones (instead of the fixed `node_type`/`region` categories).

### Recommended Use:
1. Pick a `ZONE_SET` from `ZONE_SETS` and a `GENOTYPE_SET` from `GENOTYPE_SETS`.
2. `build_zone_column()` adds a `custom_zone` column to `df_hmm` by grid_number.
3. Use `compute_state_probability()` (with `column_of_interest="custom_zone"`) to get HMM state proportions per zone.
4. Use `plot_state_probability_boxplot()` to visualize across genotypes.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_state_probability,
    plot_state_probability_boxplot,
)

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, wt_nlf, apoe_pair
ZONE_SET = "9_corners"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
STATE = 1  # Slow / Surveillance
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)
df_zone_col = build_zone_column(df_geno, ZONE_NAMES)

# Step 1: Compute proportions
state_count_df = compute_state_probability(
    df_hmm=df_zone_col,
    column_of_interest="custom_zone",
    values_displayed=ZONE_NAMES,
    state=STATE,
)

# Step 2: Plot boxplot
plot_state_probability_boxplot(
    config=config,
    state_count_df=state_count_df,
    column_of_interest="custom_zone",
    state=STATE,
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot2_state_probability",
)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_pairwise_ttests


# T-tests amongst genotypes per zone
ttest_results = run_pairwise_ttests(state_count_df, column_of_interest="custom_zone")
print(ttest_results.sort_values("p_value"))

# Plot 2b: Mean Velocity across Custom Zones

Same mechanism as Plot 2, but computes mean velocity per zone (`compute_zone_velocity()`) instead of HMM state proportion -- `plot_state_probability_boxplot()`'s `value_col`/`metric_label` parameters let it plot that instead of a state probability.

### Recommended Use:
1. Pick a `ZONE_SET` from `ZONE_SETS` and a `GENOTYPE_SET` from `GENOTYPE_SETS`.
2. `build_zone_column()` adds a `custom_zone` column to `df_hmm` by grid_number.
3. Use `compute_zone_velocity()` to get mean velocity per zone, per session.
4. Use `plot_state_probability_boxplot()` (with `value_col="velocity"`, `metric_label="Mean Velocity"`) to visualize across genotypes.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_zone_velocity,
    plot_state_probability_boxplot,
)

# 5_columns, 4_corners, six_zones, entry_reward, reward_zones, original_4, except_entry, 9_corners, 6_rows, 3_rows, 10_rows, 5_rows, wt_nlf, apoe_pair
ZONE_SET = "5_columns"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
GENOTYPE_SET = "wt_nlf"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)
df_zone_col = build_zone_column(df_geno, ZONE_NAMES)

# Step 1: Compute mean velocity per zone
velocity_zone_df = compute_zone_velocity(
    df_hmm=df_zone_col,
    column_of_interest="custom_zone",
    values_displayed=ZONE_NAMES,
)

# Step 2: Plot boxplot
plot_state_probability_boxplot(
    config=config,
    state_count_df=velocity_zone_df,
    column_of_interest="custom_zone",
    value_col="velocity",
    metric_label="Mean Velocity",
    filename_stem="mean_velocity",
    palette="Set1",
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot2b_velocity",
)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_pairwise_ttests


# T-tests amongst genotypes per zone
ttest_results_velocity = run_pairwise_ttests(velocity_zone_df, column_of_interest="custom_zone", value_col="velocity")
print(ttest_results_velocity.sort_values("p_value"))

# Plot 3: Probability of States over Time

This workflow analyzes how the probability of being in a specific HMM state evolves over time across a set of custom zones. It computes median probabilities across sessions using sliding time bins and plots one curve per zone.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- not limited to two).
2. Set a time window (`lower_limit` to `upper_limit`) and a `bin_size` to compute time bins.
3. Use `compute_node_state_medians_over_time()` with `zone_names=ZONE_NAMES` and `node_type_mapping=ZONES` to calculate median state occupancy at each zone.
4. Visualize the trajectory of state occupancy using `plot_node_state_median_curve()`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_state_medians_over_time,
    build_genotype_zone_palette,
    plot_node_state_median_curve,
)

# 5_column, 4_corner, six_zones, entry_reward, reward_zones, original_4, wt_nlf, apoe_pair
GENOTYPE_SET = "wt_nlf"
ZONE_SET = "5_rows"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)

lower_limit = 0
upper_limit = get_max_session_row_bracket(df_geno)
threshold = get_min_session_row_bracket(df_geno)  # Only show bins where all sessions are present
bin_size = 2000
figure_ylimit = (0.3, 1.1)

# zone_names takes any number of ZONES keys (not just two) -- one median curve
# per zone; node_type_mapping=ZONES looks them up in the notebook's own ZONES
# dict instead of only constants.NODE_TYPE_MAPPING's fixed categories.
deci_df = compute_node_state_medians_over_time(
    df_hmm=df_geno,
    state_types=[1],
    lower_lim=lower_limit,
    upper_lim=upper_limit,
    bin_size=bin_size,
    zone_names=ZONE_NAMES,
    node_type_mapping=ZONES,
)

# Step 2: Optional filter to only plot early session bins
deci_df = deci_df.loc[deci_df.time_bins < threshold]

# One base hue per zone; genotypes within a zone are distinguished by shades
# (light -> dark) of that same hue, instead of an arbitrary flat color per
# genotype/zone combination.
palette = build_genotype_zone_palette(deci_df)

# Step 3: Plot time-evolving median probability curves
plot_node_state_median_curve(
    config=config,
    deci_df=deci_df,
    palette=palette,
    figure_ylimit=figure_ylimit,
    fig_title=f'Median Probability of Surveillance State across {ZONE_SET} zones',
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot3_temporal_median",
)

# Plot 3a2: Probability of States over Time, One Zone per Page

Plot 3 overlays every zone's genotype curves on a single figure -- legible for `4_corner` but not for `ZONE_SET`s with more zones (e.g. `9_corners`, `6_rows`). This reuses `deci_df` from Plot 3 above and instead draws one figure per zone (hued by genotype only), collected into a single multi-page PDF via `plot_node_state_median_curves_by_zone()`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_node_state_median_curves_by_zone


# Reuses deci_df/palette/figure_ylimit/ZONE_SET/GENOTYPE_SET/FILENAME_SUFFIX from the Plot 3 cell above
plot_node_state_median_curves_by_zone(
    config=config,
    deci_df=deci_df,
    zone_names=ZONE_NAMES,
    figure_ylimit=figure_ylimit,
    palette=palette,
    fig_title="Median Probability of Surveillance State",
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot3a2_temporal_median",
)

# Plot 3b: Median Velocity over Time across Custom Zones

Same mechanism as Plot 3, but computes median per-grid-cell velocity per zone (`compute_node_velocity_medians_over_time()`) instead of HMM state proportion -- `plot_node_state_median_curve()`'s `value_col`/`metric_label` parameters let it plot that instead.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONE_SET` from `ZONE_SETS` (any number of zones -- not limited to two).
2. Set a time window (`lower_limit` to `upper_limit`) and a `bin_size` to compute time bins.
3. Use `compute_node_velocity_medians_over_time()` with `zone_names=ZONE_NAMES` and `node_type_mapping=ZONES` to calculate median velocity at each zone.
4. Visualize the trajectory of velocity using `plot_node_state_median_curve()` (with `value_col="median_velocity"`, `metric_label="Median Velocity"`).

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import (
    get_max_session_row_bracket,
    get_min_session_row_bracket,
    compute_node_velocity_medians_over_time,
    build_genotype_zone_palette,
    plot_node_state_median_curve,
)

# 5_columns, 4_corners, six_zones, entry_reward, reward_zones, original_4, except_entry, 9_corners, 6_rows, 3_rows, 10_rows, 5_rows, wt_nlf, apoe_pair
GENOTYPE_SET = "wt_nlf"
ZONE_SET = "5_rows"
ZONE_NAMES = ZONE_SETS[ZONE_SET]
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{ZONE_SET}"

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)

lower_limit = 0
upper_limit = get_max_session_row_bracket(df_geno)
threshold = get_min_session_row_bracket(df_geno)  # Only show bins where all sessions are present
bin_size = 2000
# Real per-cell mean velocity in this dataset ranges ~22-320 (see Plot 1b),
# not [0, 1] like Plot 3's state-probability figure_ylimit.
figure_ylimit = (0, 300)

# zone_names takes any number of ZONES keys (not just two) -- one median curve
# per zone; node_type_mapping=ZONES looks them up in the notebook's own ZONES
# dict instead of only constants.NODE_TYPE_MAPPING's fixed categories.
velocity_deci_df = compute_node_velocity_medians_over_time(
    df_hmm=df_geno,
    lower_lim=lower_limit,
    upper_lim=upper_limit,
    bin_size=bin_size,
    zone_names=ZONE_NAMES,
    node_type_mapping=ZONES,
)

# Step 2: Optional filter to only plot early session bins
velocity_deci_df = velocity_deci_df.loc[velocity_deci_df.time_bins < threshold]

# One base hue per zone; genotypes within a zone are distinguished by shades
# (light -> dark) of that same hue, instead of an arbitrary flat color per
# genotype/zone combination.
velocity_palette = build_genotype_zone_palette(velocity_deci_df)

# Step 3: Plot time-evolving median velocity curves
plot_node_state_median_curve(
    config=config,
    deci_df=velocity_deci_df,
    palette=velocity_palette,
    figure_ylimit=figure_ylimit,
    value_col="median_velocity",
    metric_label="Median Velocity",
    filename_stem="temporal_median_velocity_curve",
    fig_title=f'Median Velocity across {ZONE_SET} zones',
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot3b_temporal_median_velocity",
)

# Plot 3b2: Median Velocity over Time, One Zone per Page

Plot 3b overlays every zone's genotype curves on a single figure -- legible for a few zones but not for `ZONE_SET`s with more (e.g. `9_corners`, `6_rows`, `10_rows`). This reuses `velocity_deci_df` from Plot 3b above and instead draws one figure per zone (hued by genotype only), collected into a single multi-page PDF via `plot_node_state_median_curves_by_zone()`.

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import plot_node_state_median_curves_by_zone


# Reuses velocity_deci_df/velocity_palette/figure_ylimit/ZONE_SET/GENOTYPE_SET/FILENAME_SUFFIX from the Plot 3b cell above
plot_node_state_median_curves_by_zone(
    config=config,
    deci_df=velocity_deci_df,
    zone_names=ZONE_NAMES,
    figure_ylimit=figure_ylimit,
    palette=velocity_palette,
    value_col="median_velocity",
    metric_label="Median Velocity",
    filename_stem="temporal_median_velocity_curve_by_zone",
    fig_title="Median Velocity",
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot3b2_temporal_median_velocity",
)

# Plot 4: Surveillance Probability by Bout Type

This workflow evaluates behavioral surveillance patterns at a custom zone across navigational bouts in the maze. It segments the session into bouts, computes surveillance probability for each bout, and visualizes the average surveillance behavior across successful and unsuccessful bouts.

### Recommended Use:
1. Pick a `GENOTYPE_SET` and a `ZONES` key (`SURVEILLANCE_ZONE`) to compute surveillance probability at.
2. Use `assign_bout_indices_from_entry_zone()` to segment sessions into bouts.
3. Run `compute_surveillance_probabilities()` (with `node_type_mapping=ZONES`) to calculate surveillance at that zone for each bout.
4. Use `plot_surveillance_by_bout()` to generate a grouped barplot comparing surveillance probability between successful and unsuccessful bouts.

In [ ]:
from compass_labyrinth.behavior.behavior_metrics.task_performance_analysis import assign_bout_indices_from_entry_zone
from compass_labyrinth.post_hoc_analysis.level_1 import (
    compute_surveillance_probabilities,
    plot_surveillance_by_bout,
)
from compass_labyrinth.constants import ENTRY_ZONE


GENOTYPE_SET = "wt_nlf"
SURVEILLANCE_ZONE = "top_corners"
FILENAME_SUFFIX = f"_{GENOTYPE_SET}_{SURVEILLANCE_ZONE}"

df_geno = build_genotype_groups(df_hmm, GENOTYPE_SET)

# Assign Bout Numbers
# Bout = Entry zone --> Other non-entry nodes --> Entry zone
# entry_zone is [0, 1] -- delimiter_nodes=ENTRY_ZONE catches sessions that
# never happen to have tracked data in cell 0 exactly but do in its neighbor
# cell 1 (see task_performance_analysis for the original fix).
df_geno = assign_bout_indices_from_entry_zone(
    df_geno,
    delimiter_nodes=ENTRY_ZONE,
)

# Compute surveillance probability at SURVEILLANCE_ZONE by Bout type
# node_type_mapping=ZONES lets decision_nodes be any ZONES key, instead of
# only constants.NODE_TYPE_MAPPING's fixed categories.
# Successful -> reached the target at least once / Unsuccessful -> didn't
index_df, median_df = compute_surveillance_probabilities(
    df_hmm=df_geno,
    decision_nodes=SURVEILLANCE_ZONE,
    node_type_mapping=ZONES,
)

# Barplot to depict the above with ttest-ind pvalue
plot_surveillance_by_bout(
    config=config,
    median_df=median_df,
    ylim=1,
    filename_suffix=FILENAME_SUFFIX,
    figures_dirname="figures_lvl1_b/plot4_surveillance_bout",
)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import run_within_genotype_mixedlm_with_fdr


# LMM for same genotype comparison across Bout types
df_within = run_within_genotype_mixedlm_with_fdr(index_df)

# Print results
print("Within-Genotype (Successful vs Unsuccessful):")
print(df_within)

In [ ]:
from compass_labyrinth.post_hoc_analysis.level_1 import test_across_genotypes_per_bout


# T-test across genotypes under Successful Bouts
df_across_success = test_across_genotypes_per_bout(median_df, bout_type='Successful')

# T-test across genotypes under Unsuccessful Bouts
df_across_unsuccess = test_across_genotypes_per_bout(median_df, bout_type='Unsuccessful')

# Print results
print("\n Across Genotypes (Successful only):")
print(df_across_success)

print("\n Across Genotypes (Unsuccessful only):")
print(df_across_unsuccess)